# M04 — Turn a Messy CSV into Usable Data

**Objective:** clean and transform imperfect tabular data for analysis without hiding defects, decisions, or uncertainty.

This lab is CPU-only, deterministic, secret-free, paid-API-free, and network-free.

## Workflow

**raw → inspect → predict defects → declare quality expectations → duplicate analysis → normalization → numeric/date parsing → missingness decisions → validate constraints → investigate outliers → reproducible cleaning pipeline → raw-vs-clean comparison → analysis-ready table → invariant assertions**

## 1. Setup

**Predict before running:** Why should a dirty CSV be loaded as strings before pandas infers types? Record one kind of evidence type inference could destroy.

In [ ]:
from pathlib import Path
import sys

root_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((candidate for candidate in root_candidates if (candidate / "datasets" / "M04" / "customer_orders_dirty.csv").is_file()), None)
assert ROOT is not None, "Could not locate the LearningOS-AI repository root."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from missions.M04.cleaning import (
    ALLOWED_CATEGORIES,
    ALLOWED_REGIONS,
    ALLOWED_STATUSES,
    DATE_FORMATS,
    MAX_ORDER_DATE,
    MIN_ORDER_DATE,
    assert_analysis_ready,
    clean_orders,
    load_raw,
    raw_vs_clean_comparison,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 90)
DATA_PATH = ROOT / "datasets" / "M04" / "customer_orders_dirty.csv"

In [ ]:
raw = load_raw(DATA_PATH)
print(f"raw shape: {raw.shape}")
display(raw.head(8))
assert raw.shape == (36, 11)
assert all(dtype.name == "string" for dtype in raw.dtypes)

## 2. Inspect the raw state

**Predict before running:** Which columns will have many spellings for the same meaning? Which columns should not yet be trusted as numeric or dates?

In [ ]:
raw_profile = pd.DataFrame({
    "blank_count": raw.eq("").sum(),
    "distinct_raw_values": raw.nunique(dropna=False),
    "dtype": raw.dtypes.astype(str),
})
display(raw_profile)
print("raw region spellings:", sorted(raw["region"].unique().tolist()))
print("raw category spellings:", sorted(raw["category"].unique().tolist()))

## 3. Predict defects before transformation

Do not run later cleaning cells until you record: column, predicted defect, expected evidence, possible impact, and confidence. Predictions are hypotheses—not findings. After inspection, compare them with the issue summary.

### Use precise evidence language

| Term | Meaning | Example |
|---|---|---|
| Observed defect | Directly visible value or failed rule | `units=two` |
| Hypothesis | Explanation that still needs testing | data-entry word instead of count |
| Decision | Declared treatment | keep row in review; do not guess |
| Evidence | Raw value, rule, trace, or owner note | raw source row and parser result |
| Lost information / uncertainty | What cannot yet be recovered | intended unit count is unknown |

## 4. Declare quality expectations

The data contract precedes cleaning: IDs follow `ORD-####` and are unique in analysis-ready data; domains are enumerated; units are positive whole numbers; money is positive; dates use declared formats and range; email is structurally valid; totals reconcile to units × price; required fields are present.

In [ ]:
contract = {
    "regions": sorted(ALLOWED_REGIONS),
    "categories": sorted(ALLOWED_CATEGORIES),
    "statuses": sorted(ALLOWED_STATUSES),
    "accepted_date_formats": DATE_FORMATS,
    "date_range": (str(MIN_ORDER_DATE.date()), str(MAX_ORDER_DATE.date())),
    "total_rule": "abs(order_total - units * unit_price) <= 0.01",
}
contract

## 5. Duplicate analysis

**Predict before running:** How many rows are exact duplicates? Which IDs may still conflict after exact duplicates are removed? Why is `drop_duplicates(subset="order_id")` unsafe?

In [ ]:
exact_duplicate_rows = raw.loc[raw.duplicated(keep=False)]
display(exact_duplicate_rows)
assert raw.duplicated(keep="first").sum() == 1

In [ ]:
normalized_ids = raw["order_id"].str.strip().str.upper()
normalized_id_counts = normalized_ids[normalized_ids.ne("")].value_counts()
duplicate_id_values = normalized_id_counts[normalized_id_counts > 1].index
display(raw.loc[normalized_ids.isin(duplicate_id_values)].assign(normalized_id=normalized_ids))
assert set(duplicate_id_values) == {"ORD-1003", "ORD-1005", "ORD-1012"}

**Decision:** remove only the second byte-equivalent record into an audit log. Preserve all non-identical records sharing a normalized ID and mark them as conflicts. The duplicate row's evidence is not lost because its source row is retained in `duplicate_log`.

## 6. Normalization

**Predict before running:** What canonical values should result from ` north `, `N`, `home/kitchen`, `home and kitchen`, `DONE`, and `in progress`? What raw evidence must remain?

In [ ]:
result = clean_orders(raw)
print(result.summary)
assert len(result.cleaned) == 35
assert len(result.duplicate_log) == 1

In [ ]:
normalization_columns = [
    "source_row", "raw_order_id", "order_id",
    "raw_customer_name", "customer_name",
    "raw_region", "region", "raw_category", "category",
    "raw_status", "status",
]
display(result.cleaned.loc[result.cleaned["source_row"].isin([3, 6, 15, 30]), normalization_columns])
assert result.cleaned.loc[result.cleaned["source_row"].eq(3), "order_id"].iat[0] == "ORD-1002"

## 7. Numeric and date parsing

**Predict before running:** Which currency strings should parse? Which malformed strings and impossible dates must produce issue evidence instead of a silent missing value?

In [ ]:
numeric_columns = [
    "order_id", "raw_units", "units", "raw_unit_price",
    "unit_price", "raw_order_total", "order_total", "observed_defects",
]
display(result.cleaned.loc[result.cleaned["order_id"].isin(["ORD-1001", "ORD-1010", "ORD-1011", "ORD-1013", "ORD-1024"]), numeric_columns])

In [ ]:
date_columns = ["order_id", "raw_order_date", "order_date", "observed_defects", "decision_log", "uncertainty"]
date_examples = ["ORD-1002", "ORD-1003", "ORD-1004", "ORD-1012", "ORD-1016", "ORD-1018"]
display(result.cleaned.loc[result.cleaned["order_id"].isin(date_examples), date_columns])

## 8. Missingness decisions

**Predict before running:** Which blank can be derived from direct evidence? Which blanks must remain unresolved? Distinguish missing from malformed before deciding.

In [ ]:
missingness_ids = ["ORD-1006", "ORD-1007", "ORD-1013", "ORD-1019", "ORD-1023", "ORD-1031"]
missingness_columns = ["order_id", "raw_order_total", "order_total", "observed_defects", "decision_log", "analysis_ready"]
display(result.cleaned.loc[result.cleaned["order_id"].isin(missingness_ids), missingness_columns])
derived = result.cleaned.loc[result.cleaned["order_id"].eq("ORD-1013")].iloc[0]
assert derived["order_total"] == 1600
assert "derive_missing_total" in derived["decision_log"]

## 9. Validate constraints

**Predict before running:** Which failures are blocking for analysis? Which non-blocking issue records a policy or uncertainty without making the row numerically unusable?

In [ ]:
display(result.issue_summary)
print("analysis-ready rows:", len(result.analysis_ready))
print("review rows:", (~result.cleaned["analysis_ready"]).sum())

In [ ]:
constraint_columns = [
    "source_row", "order_id", "units", "unit_price", "order_total",
    "order_date", "observed_defects", "analysis_ready",
]
display(result.cleaned.loc[~result.cleaned["analysis_ready"], constraint_columns])

## 10. Investigate outliers

**Predict before running:** Which extreme value is supported by business evidence? Which extreme also violates arithmetic reconciliation? A statistical flag is evidence for investigation, not permission to delete.

In [ ]:
thresholds = {key: value for key, value in result.summary.items() if key.endswith("_outlier_threshold")}
thresholds

In [ ]:
outlier_columns = [
    "order_id", "units", "unit_price", "order_total", "notes",
    "outlier_reason", "outlier_judgment", "observed_defects", "analysis_ready",
]
outliers = result.cleaned.loc[result.cleaned["outlier_flag"], outlier_columns]
display(outliers)
assert "ORD-1028" in set(outliers["order_id"])
assert result.cleaned.loc[result.cleaned["order_id"].eq("ORD-1028"), "analysis_ready"].iat[0]
assert result.cleaned.loc[result.cleaned["order_id"].eq("ORD-1028"), "outlier_judgment"].iat[0] == "retain_business_exception"

## 11. Controlled failure — unsafe cleaning

Predict the evidence loss before running each shortcut. The failure is measured and caught so Restart + Run All remains successful.

In [ ]:
unsafe_dropna = raw.replace("", pd.NA).dropna()
print("blanket dropna rows:", len(unsafe_dropna), "of", len(raw))
print("rows lost:", len(raw) - len(unsafe_dropna))
assert len(unsafe_dropna) < len(raw)

In [ ]:
unsafe_normalized = raw.assign(order_id=raw["order_id"].str.strip().str.upper())
unsafe_dedupe = unsafe_normalized.drop_duplicates(subset="order_id", keep="first")
print("aggressive ID dedupe rows lost:", len(raw) - len(unsafe_dedupe))
print("lost conflict IDs include ORD-1005 and ORD-1012")
assert len(raw) - len(unsafe_dedupe) == 3

In [ ]:
unsafe_price_text = raw["unit_price"].str.replace(r"[₹$£€,\s]", "", regex=True)
unsafe_price = pd.to_numeric(unsafe_price_text, errors="coerce")
silent_new_missing = unsafe_price.isna() & raw["unit_price"].ne("")
display(raw.loc[silent_new_missing, ["order_id", "unit_price"]])
assert silent_new_missing.sum() == 1

In [ ]:
unsafe_without_outliers = result.cleaned.loc[~result.cleaned["outlier_flag"]]
print("automatic outlier deletion rows lost:", len(result.cleaned) - len(unsafe_without_outliers))
assert "ORD-1028" not in set(unsafe_without_outliers["order_id"])
print("Approved wholesale evidence would have been deleted.")

### Diagnose the controlled failure

For each shortcut record: observed loss, hypothesis about why it occurred, implicit decision, evidence erased, lost information or uncertainty, and the smallest safe repair. Use `missions/M04/controlled_failure.md` as the review checklist.

## 12. Reproducible cleaning pipeline

**Predict before running:** If no source or policy changes, which outputs must be identical on a second run?

In [ ]:
second_result = clean_orders(load_raw(DATA_PATH))
pd.testing.assert_frame_equal(result.cleaned, second_result.cleaned)
pd.testing.assert_frame_equal(result.duplicate_log, second_result.duplicate_log)
pd.testing.assert_frame_equal(result.issue_summary, second_result.issue_summary)
assert result.summary == second_result.summary
print("deterministic rerun: PASS")

## 13. Raw-vs-clean comparison

**Predict before running:** Verify the row equation `raw = cleaned + exact duplicates logged`. Ready plus review must equal cleaned. Outliers must remain inside cleaned.

In [ ]:
comparison = raw_vs_clean_comparison(result)
display(comparison)
assert len(result.raw) == len(result.cleaned) + len(result.duplicate_log)
assert len(result.cleaned) == len(result.analysis_ready) + (~result.cleaned["analysis_ready"]).sum()
assert result.cleaned["outlier_flag"].sum() == result.summary["outliers_retained"]

## 14. Analysis-ready table

The ready table is an explicit contract view, not evidence deletion. Review rows remain in `result.cleaned`, and exact duplicates remain in `result.duplicate_log`.

In [ ]:
analysis_columns = [
    "order_id", "customer_name", "region", "category", "units",
    "unit_price", "order_total", "order_date", "status",
    "outlier_flag", "outlier_judgment",
]
display(result.analysis_ready[analysis_columns])
print(f"analysis-ready shape: {result.analysis_ready.shape}")

## 15. Invariant assertions

**Predict before running:** Name one representative and one boundary case for identity, numeric, domain, date and reconciliation invariants.

In [ ]:
assert assert_analysis_ready(result.analysis_ready)
assert result.analysis_ready["order_id"].is_unique
assert result.analysis_ready["units"].gt(0).all()
assert result.analysis_ready["units"].mod(1).eq(0).all()
assert set(result.analysis_ready["region"]).issubset(ALLOWED_REGIONS)
assert set(result.analysis_ready["category"]).issubset(ALLOWED_CATEGORIES)
assert result.analysis_ready["order_date"].between(MIN_ORDER_DATE, MAX_ORDER_DATE).all()
assert (result.analysis_ready["order_total"] - result.analysis_ready["units"] * result.analysis_ready["unit_price"]).abs().le(0.01).all()
print("analysis-ready invariant assertions: PASS")

## 16. Evidence language check

Choose three rows and write five separate statements: observed defect, hypothesis, decision, evidence, and lost information / uncertainty. Do not call a hypothesis a fact. Do not call a statistical flag an error.

## 17. Code reading

Complete `missions/M04/code_reading.md`. Trace one currency row and one conflicting-ID row through `clean_orders` before editing the function.

## 18. No-AI Gate

Complete `missions/M04/no_ai_gate.md` on the fresh inventory fixture **without AI-generated code**. The gate tests transfer, row accounting, validation and uncertainty communication.

## 19. Formal engineering review

Use `missions/M04/review_brief.md`. Present raw state, data contract, architecture, evidence, controlled failure, validation, uncertainty / information loss, and V01 integration. A clean-looking table is not sufficient evidence.

## 20. Assessment

Submit the evidence defined in `missions/M04/evidence_contract.yaml`. Passing means the pipeline is reproducible and auditable, the analysis-ready invariants pass, unsafe cleaning is diagnosed from evidence, uncertainty remains visible, and the learner transfers the method to fresh data.